## Library

In [ ]:
# Data Processing & File Management
import os
import glob
import pickle
import pathlib
import datetime
from datetime import datetime
from time import time

# Data Retrieval & I/O
import pandas as pd
import dask.dataframe as dd
from dataretrieval import nwis
import earthaccess
import requests
import netCDF4 as nc
from dask.distributed import Client, progress
import time
from tqdm import tqdm

# Geospatial & Visualization
import geopandas as gpd
import matplotlib.pyplot as plt

# Numerical Computing
import numpy as np

# Performance Optimization
import gc
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Jupyter Display
from IPython.display import clear_output
import requests
import time
import subprocess
import re
import rasterio
from io import BytesIO
from pyproj import CRS, Transformer
import subprocess

## Download Gage Height (00065) for SWOT Version D from 2025/04/01 with Merritt's Gage Info

In [ ]:
usgs_link_merritt = pd.read_csv(
    "/work/pi_kandread_umass_edu/swot-urban/0_data_collection/5_sword_usgs/paired_reach_SWOT_gage.csv", 
    dtype={'gage_id': object}
)

target_sites = usgs_link_merritt[
    usgs_link_merritt.gage_agency == "USGS"
].gage_id.drop_duplicates().tolist()



In [ ]:
# ============================================================================
# 2. Download site information (datum, location, etc.)
# ============================================================================
print("\n" + "="*80)
print("Downloading site information...")
print("="*80)

site_info_parts = []
batch_size = 100  # NWIS API limit per request

for i in range(0, len(target_sites), batch_size):
    batch = target_sites[i:i+batch_size]
    clear_output(wait=True)
    print(f"Downloading site info: {i+1}-{min(i+batch_size, len(target_sites))} of {len(target_sites)}")
    
    try:
        # Get site information - returns tuple (df, md)
        result = nwis.get_info(sites=batch)
        
        # Handle different return types
        if isinstance(result, tuple):
            site_batch = result[0]  # First element is the dataframe
        else:
            site_batch = result
        
        # Check if we got valid data
        if site_batch is not None and isinstance(site_batch, pd.DataFrame) and not site_batch.empty:
            site_info_parts.append(site_batch)
            print(f"  → Downloaded: {len(site_batch)} sites")
        else:
            print(f"  → No data in this batch")
            
    except Exception as e:
        print(f"  → Error in batch {i}: {str(e)}")
        continue
    
    time.sleep(0.5)

# Combine all site info
if site_info_parts:
    all_site_info = pd.concat(site_info_parts, axis=0, ignore_index=False)
    print(f"\nTotal sites with info: {len(all_site_info)}")
else:
    print("\nWarning: No site information downloaded!")
    all_site_info = pd.DataFrame()

#### Download script and save as pickle

In [ ]:
# ============================================================================
# 3. Download WSE time series data
# ============================================================================
print("\n" + "="*80)
print("Downloading WSE time series...")
print("="*80)

START, END = "2023-01-01", "2025-10-31"

all_wse_parts = []
failed_sites = []
success_count = 0

for idx, site in enumerate(target_sites, 1):
    clear_output(wait=True)
    print(f"Processing site {idx}/{len(target_sites)}: {site}")
    print(f"Success so far: {success_count}")
    
    try:
        # Download instantaneous values (15-min interval)
        # Parameter codes: 62617 (NAVD88, meters), 62622 (NAVD88, meters)
        result = nwis.get_iv(
            sites=site,
            startDT=START, 
            endDT=END,
            parameterCd="00065"
        )
        
        # Handle tuple return
        if isinstance(result, tuple):
            df = result[0]
        else:
            df = result
        
        if df is not None and isinstance(df, pd.DataFrame) and not df.empty:
            # Reset index to get datetime as column
            df = df.reset_index()
            all_wse_parts.append(df)
            success_count += 1
            print(f"  → Success: {len(df)} records")
        else:
            print(f"  → No data available")
            failed_sites.append(site)
            
    except Exception as e:
        print(f"  → Error: {str(e)}")
        failed_sites.append(site)
        continue
    
    time.sleep(0.3)  # Rate limiting

In [ ]:
with open('/work/pi_kandread_umass_edu/swot-urban/1_data_processing/all_wse_parts.pkl', 'wb') as f:
    pickle.dump(all_wse_parts, f)

#### Check Identical with pickle file

In [ ]:
if len(all_wse_parts) != len(all_wse_parts_p):
    print("The lists have different lengths.")
else:
    for i, (orig, loaded) in enumerate(zip(all_wse_parts, all_wse_parts_p)):
        # Use DataFrame.equals() method for comparison if elements are DataFrames
        if hasattr(orig, 'equals') and callable(orig.equals):
            if not orig.equals(loaded):
                print(f"Difference found at index {i}: original and loaded DataFrames are different")
                break
        else:
            if orig != loaded:
                print(f"Difference found at index {i}: original={orig}, loaded from pickle={loaded}")
                break
    else:
        print("All elements are identical.")

#### Concat downloaded list to dataframe

In [ ]:
with open('/work/pi_kandread_umass_edu/swot-urban/1_data_processing/all_wse_parts.pkl', 'rb') as f:
    all_wse_parts = pickle.load(f)

In [ ]:
# ============================================================================
# Setup Dask Distributed Client for Parallel Processing
# ============================================================================

# Initialize Dask client (adjust workers based on your system)
client = Client(n_workers=12, threads_per_worker=4, memory_limit='128GB')
print(f"✓ Dask Dashboard: {client.dashboard_link}")
print(f"  Workers: {len(client.scheduler_info()['workers'])}")

In [ ]:
# ============================================================================
# Process and combine gage height data
# ============================================================================

# Convert to Dask (not yet loaded into memory)
dask_dfs = []
for i, df_part in enumerate(all_wse_parts):
    dask_df = dd.from_pandas(df_part, npartitions=1)
    dask_dfs.append(dask_df)
    all_wse_parts[i] = None
    
    if i % 50 == 0:
        gc.collect()

# Concatenate with Dask (still not loaded into memory)
all_gage_dask = dd.concat(dask_dfs, axis=0)

In [ ]:
# Rename datetime column
if 'datetime' in all_gage_dask.columns:
    all_gage_dask = all_gage_dask.rename(columns={'datetime': 'measurement_dt'})
all_gage_dask['measurement_dt'] = dd.to_datetime(all_gage_dask['measurement_dt'])

# Find the gage height value column (usually named like '00065' or '00065_Mean')
gage_col = [col for col in all_gage_dask.columns if '00065' in col and 'cd' not in col.lower()][0]
all_gage_dask = all_gage_dask.rename(columns={gage_col: 'gage_height_ft'})

# Drop all quality code columns (columns ending with '_cd')
cd_columns = [col for col in all_gage_dask.columns if col.lower().endswith('_cd')]
if cd_columns:
    all_gage_dask = all_gage_dask.drop(columns=cd_columns)
    print(f"Dropped {len(cd_columns)} quality code columns (_cd)")
else:
    print("No quality code columns (_cd) found to drop")

In [ ]:
print("="*80)
print("Missing Gage Height Data Restructuring (Dask version)")
print("="*80)

# Step 1: Identify 00065_* columns
print("\nStep 1: Identifying 00065_* columns...")

all_columns = all_gage_dask.columns.tolist()
gage_height_columns = [col for col in all_columns if col.startswith('00065_')]

print(f"Total 00065_* columns found: {len(gage_height_columns)}")

# Step 2: Exclude datum-related columns
print("\nStep 2: Filtering out datum-related columns...")

datum_keywords = [
    'navd88', 'navd 1988', 'igld85', 'igld 85', 
    'nws datum', 'usgs/nws datum',
    'elevation referenced to', 'ism'
]

columns_to_exclude = []
for col in gage_height_columns:
    col_lower = col.lower()
    if any(keyword in col_lower for keyword in datum_keywords):
        columns_to_exclude.append(col)

print(f"Columns to exclude (datum-related): {len(columns_to_exclude)}")
for col in sorted(columns_to_exclude):
    print(f"  - {col}")

working_columns = [col for col in gage_height_columns if col not in columns_to_exclude]
print(f"\nWorking with {len(working_columns)} columns after exclusion")

# Step 3: Identify paired columns
print("\nStep 3: Identifying paired columns (upstream/downstream, headwater/tailwater)...")

paired_columns = {
    'upstream_downstream': {'left': [], 'right': []},
    'headwater_tailwater': {'left': [], 'right': []},
}

unpaired_columns = []

for col in working_columns:
    col_lower = col.lower()
    if 'upstream' in col_lower:
        paired_columns['upstream_downstream']['left'].append(col)
    elif 'downstream' in col_lower or 'down stream' in col_lower:
        paired_columns['upstream_downstream']['right'].append(col)
    elif 'headwater' in col_lower:
        paired_columns['headwater_tailwater']['left'].append(col)
    elif 'tailwater' in col_lower:
        paired_columns['headwater_tailwater']['right'].append(col)
    else:
        unpaired_columns.append(col)

print(f"\nPaired columns found:")
for pair_type, cols in paired_columns.items():
    print(f"  {pair_type}:")
    print(f"    Left ({len(cols['left'])}): {cols['left']}")
    print(f"    Right ({len(cols['right'])}): {cols['right']}")

print(f"\nUnpaired columns ({len(unpaired_columns)}): will go to gage_height_missing")

# Step 4: Define processing function
print("\nStep 4: Creating processing function...")

def process_partition(partition_df, unpaired_cols, paired_cols):
    """Process each partition with pandas operations"""
    # Create new columns
    result_df = partition_df[['measurement_dt', 'site_no', 'gage_height_ft']].copy()
    result_df['gage_height_missing'] = np.nan
    result_df['gage_height_missing_left'] = np.nan
    result_df['gage_height_missing_right'] = np.nan
    result_df['source_column'] = ''
    
    # Process unpaired columns
    for col in unpaired_cols:
        if col not in partition_df.columns:
            continue
        mask = partition_df[col].notna()
        no_value_yet = result_df['gage_height_missing'].isna()
        update_mask = mask & no_value_yet
        if update_mask.sum() > 0:
            result_df.loc[update_mask, 'gage_height_missing'] = partition_df.loc[update_mask, col]
            result_df.loc[update_mask, 'source_column'] = col
    
    # Process paired columns - left (upstream/headwater)
    all_left_cols = paired_cols['upstream_downstream']['left'] + paired_cols['headwater_tailwater']['left']
    for left_col in all_left_cols:
        if left_col not in partition_df.columns:
            continue
        mask = partition_df[left_col].notna()
        no_value_yet = result_df['gage_height_missing_left'].isna()
        update_mask = mask & no_value_yet
        if update_mask.sum() > 0:
            result_df.loc[update_mask, 'gage_height_missing_left'] = partition_df.loc[update_mask, left_col]
            for idx in result_df[update_mask].index:
                existing = result_df.loc[idx, 'source_column']
                new_source = f"left:{left_col}"
                result_df.loc[idx, 'source_column'] = f"{existing},{new_source}" if existing else new_source
    
    # Process paired columns - right (downstream/tailwater)
    all_right_cols = paired_cols['upstream_downstream']['right'] + paired_cols['headwater_tailwater']['right']
    for right_col in all_right_cols:
        if right_col not in partition_df.columns:
            continue
        mask = partition_df[right_col].notna()
        no_value_yet = result_df['gage_height_missing_right'].isna()
        update_mask = mask & no_value_yet
        if update_mask.sum() > 0:
            result_df.loc[update_mask, 'gage_height_missing_right'] = partition_df.loc[update_mask, right_col]
            for idx in result_df[update_mask].index:
                existing = result_df.loc[idx, 'source_column']
                new_source = f"right:{right_col}"
                result_df.loc[idx, 'source_column'] = f"{existing},{new_source}" if existing else new_source
    
    return result_df

# Step 5: Apply processing to create lazy computation graph
print("\nStep 5: Creating computation graph...")

meta = pd.DataFrame({
    'measurement_dt': pd.Series(dtype='datetime64[ns]'),
    'site_no': pd.Series(dtype='object'),
    'gage_height_ft': pd.Series(dtype='float64'),
    'gage_height_missing': pd.Series(dtype='float64'),
    'gage_height_missing_left': pd.Series(dtype='float64'),
    'gage_height_missing_right': pd.Series(dtype='float64'),
    'source_column': pd.Series(dtype='object')
})

all_gage_miss_dask = all_gage_dask.map_partitions(
    process_partition,
    unpaired_columns,
    paired_columns,
    meta=meta
)

print(f"Computation graph created with {all_gage_miss_dask.npartitions} partitions")

In [ ]:
# Convert all feet-based gage height columns to meters and rename columns
# All gage_height columns are in feet - convert to meters
ft_cols = ['gage_height_ft', 'gage_height_missing', 
           'gage_height_missing_left', 'gage_height_missing_right']

for col in ft_cols:
    all_gage_miss_dask[col] = all_gage_miss_dask[col] * 0.3048
    print(f"✓ Converted {col} to meters")

# Rename gage_height_ft to gage_height_m
all_gage_miss_dask = all_gage_miss_dask.rename(columns={'gage_height_ft': 'gage_height_m'})
print("\n✓ Renamed gage_height_ft → gage_height_m")

print("\nFinal columns:", list(all_gage_miss_dask.columns))

#### sites over 90k samples

In [ ]:
# ============================================================================
# Count valid observations with fallback hierarchy
# ============================================================================
print("\n" + "="*80)
print("Counting valid observations (with fallback hierarchy)")
print("="*80)
print("Priority: gage_height_m → gage_height_missing → (left+right)/2")

# Create validity mask with fallback hierarchy (using isna + ~)
valid_obs_mask = (
    ~all_gage_miss_dask['gage_height_m'].isna() |  # Primary: gage_height_m
    ~all_gage_miss_dask['gage_height_missing'].isna() |  # Fallback 1
    (~all_gage_miss_dask['gage_height_missing_left'].isna() & 
     ~all_gage_miss_dask['gage_height_missing_right'].isna())  # Fallback 2: both must exist
)

# Add validity column to dataframe
all_gage_miss_dask['valid_obs'] = valid_obs_mask.astype(int)

# Count valid observations per site
print("\nComputing statistics (this may take a moment)...")
site_valid_counts = all_gage_miss_dask.groupby('site_no')['valid_obs'].sum().compute()

print(f"\n✓ Total sites: {len(site_valid_counts):,}")
print(f"✓ Total valid observations: {site_valid_counts.sum():,.0f}")

# Show describe
print("\n" + "="*80)
print("Distribution of valid observations per site:")
print("="*80)
print(site_valid_counts.describe())

# Get sites with >= 90,000 observations
sites_over_90k = site_valid_counts[site_valid_counts >= 90000].index.tolist()

print(f"\n" + "="*80)
print(f"✓ Sites with ≥90,000 valid observations: {len(sites_over_90k):,}")
print("="*80)

# Save to pickle
with open('sites_over_90k.pkl', 'wb') as f:
    pickle.dump(sites_over_90k, f)
print(f"✓ Saved to 'sites_over_90k.pkl'")

# Show sample of sites
print(f"\nSample sites: {sites_over_90k[:5]}...")

In [ ]:
# Save to pickle
with open('sites_over_90k.pkl', 'wb') as f:
    pickle.dump(sites_over_90k, f)

print(f"✓ Saved {len(sites_over_90k):,} sites to 'sites_over_90k.pkl'")

### Merge buffered polygons with usgs gage heights

In [ ]:
# ============================================================================
# Load target sites (USGS gages linked to SWORD)
# ============================================================================
usgs_link_merritt = pd.read_csv(
    "/work/pi_kandread_umass_edu/swot-urban/0_data_collection/5_sword_usgs/paired_reach_SWOT_gage.csv", 
    dtype={'gage_id': object}
)

target_sites = usgs_link_merritt[
    usgs_link_merritt.gage_agency == "USGS"
].gage_id.drop_duplicates().tolist()

# ============================================================================
# Download site information (datum, location, etc.)
# ============================================================================
print("\n" + "="*80)
print("Downloading site information...")
print("="*80)

site_info_parts = []
batch_size = 100  # NWIS API limit per request

for i in range(0, len(target_sites), batch_size):
    batch = target_sites[i:i+batch_size]
    clear_output(wait=True)
    print(f"Downloading site info: {i+1}-{min(i+batch_size, len(target_sites))} of {len(target_sites)}")
    
    try:
        # Get site information - returns tuple (df, md)
        result = nwis.get_info(sites=batch)
        
        # Handle different return types
        if isinstance(result, tuple):
            site_batch = result[0]  # First element is the dataframe
        else:
            site_batch = result
        
        # Check if we got valid data
        if site_batch is not None and isinstance(site_batch, pd.DataFrame) and not site_batch.empty:
            site_info_parts.append(site_batch)
            print(f"  → Downloaded: {len(site_batch)} sites")
        else:
            print(f"  → No data in this batch")
            
    except Exception as e:
        print(f"  → Error in batch {i}: {str(e)}")
        continue
    
    time.sleep(0.5)

# Combine all site info
if site_info_parts:
    all_site_info = pd.concat(site_info_parts, axis=0, ignore_index=False)
    print(f"\nTotal sites with info: {len(all_site_info)}")
else:
    print("\nWarning: No site information downloaded!")
    all_site_info = pd.DataFrame()

In [ ]:
all_site_info['alt_va_m'] = all_site_info.alt_va * 0.3048

In [ ]:
with open('sites_over_90k.pkl', 'rb') as f:
    sites_over_90k = pickle.load(f)

In [ ]:
# Read the parquet file with NLCD data
buffered_polygons = gpd.read_parquet(
    "/work/pi_kandread_umass_edu/swot-urban/1_data_processing/buffered_polygons_with_nlcd.parquet"
)

# # Read the parquet file with usgs gage heights
# usgs_height= pd.read_parquet("/work/pi_kandread_umass_edu/swot-urban/1_data_processing/usgs_gages/usgs_gage_height_23_25.parquet")

In [ ]:
# NLCD sum/mean pairs
for year in [21, 22, 23, 24]:
    sum_col = f'nlcd_{year}_sum'
    mean_col = f'nlcd_{year}_mean'
    buffered_polygons.loc[buffered_polygons[sum_col] == 0, mean_col] = 0

# fctimp count/mean pair
buffered_polygons.loc[buffered_polygons['fctimp_count'] == 0, 'fctimp_mean'] = 0

### Download SWOT Sword of Science River Discharge Products Version 1 and check if it has usgs reach_id and site_no

In [ ]:
# Get the access
earthaccess.login()

# Search for one granule
granule_info = earthaccess.search_data(
    short_name="SWOT_L4_DAWG_SOS_DISCHARGE",
    temporal=("2024-01-01", "2025-11-03"),
    # count=1
)

print(f"Found {len(granule_info)} granule(s)")

if granule_info:
    # Download the priors file
    print("\nDownloading priors file...")
    files = earthaccess.download(
        granule_info,
        local_path="/work/pi_kandread_umass_edu/swot-urban/0_data_collection/5_sword_usgs/"
    )

In [ ]:
import netCDF4 as nc
import numpy as np
import pandas as pd
import glob
import os

# Find the priors file
priors_dir = "/work/pi_kandread_umass_edu/swot-urban/0_data_collection/5_sword_usgs/"
priors_files = glob.glob(os.path.join(priors_dir, "na*0726*priors*.nc"))

if not priors_files:
    print(f"No files found")
    exit(1)

priors_file = priors_files[0]
print(f"File: {os.path.basename(priors_file)}\n")

# Open the netCDF file
ds = nc.Dataset(priors_file, 'r')

# Check what groups are available
print("Groups in this file:")
for group_name in ds.groups.keys():
    print(f"  - {group_name}")

# Look at the USGS group
if 'USGS' in ds.groups: 
    usgs = ds.groups['USGS']
    
    # Check dimensions
    print("\nDimensions:")
    for dim_name, dim in usgs.dimensions.items():
        print(f"  {dim_name}: {len(dim)}")
    
    # Check what variables are in USGS group
    print("\nVariables in USGS group:")
    for var_name in usgs.variables.keys():
        var = usgs.variables[var_name]
        print(f"  {var_name}: {var.shape}, {var.dtype}")
    
    # Convert USGS data to DataFrame
    # Get the main variables
    reach_ids = usgs.variables['USGS_reach_id'][:]
    
    # USGS_id is stored as char array, need to convert to strings
    usgs_id_chars = usgs.variables['USGS_id'][:]
    usgs_site_ids = []
    for row in usgs_id_chars:
        site_id = ''.join([c.decode('utf-8') if isinstance(c, bytes) else c for c in row]).strip()
        usgs_site_ids.append(site_id)
    
    # Create DataFrame
    usgs_df = pd.DataFrame({'reach_id': reach_ids, 'usgs_site_no': usgs_site_ids})
    
    # Save as parquet
    output_path = "/work/pi_kandread_umass_edu/swot-urban/0_data_collection/5_sword_usgs/usgs_sword_linkage.parquet"
    usgs_df.to_parquet(output_path, index=False)
    
    print(f"Saved to: {output_path}")
    print(f"Shape: {usgs_df.shape}")
ds.close()

### Link USGS gage heights and Buffered river width polygons

In [ ]:
# Read the linkage file
usgs_link_merritt = pd.read_csv("/work/pi_kandread_umass_edu/swot-urban/0_data_collection/5_sword_usgs/paired_reach_SWOT_gage.csv", dtype={'gage_id': object})

# Rename usgs_site_no to site_no
usgs_link_merritt = usgs_link_merritt.rename(columns={'gage_id': 'site_no'})

print(f"\nusgs_link shape: {usgs_link_merritt.shape}")
print(f"buffered_polygons shape: {buffered_polygons.shape}")

In [ ]:
usgs_merritt = usgs_link_merritt[usgs_link_merritt.site_no.isin(sites_over_90k)]\
[['site_no', 'gage_name', 'gage_latitude','gage_longitude', 'gage_coord_datum', 'gage_alt_datum',\
  'gage_time_UTC', 'gage_stage_m', 'reach_id', 'SWOT_time','dark_frac', 'ice_clim_f', 'partial_f',\
  'n_good_nod', 'xovr_cal_q', 'p_length', 'p_width', 'p_wse','p_n_ch_mod', 'p_dam_id', 'p_n_nodes',\
  'p_low_slp', 'xtrk_dist','sword_version', 'collection_shortname', 'cycle_id', 'pass_id', 'crid']]
merritt_reach_id = usgs_merritt.reach_id.to_list()
merritt_buffered = buffered_polygons[buffered_polygons.reach_id.isin(merritt_reach_id)]

merritt_df = merritt_buffered.merge(usgs_merritt, on='reach_id', how='inner')
print(f"buffered_polygons + usgs_link_merritt merged dataframe shape: {merritt_df .shape}")

# Save as parquet for later spatial mapping with GeoPandas (GeoDataFrame)
output_path = "/work/pi_kandread_umass_edu/swot-urban/1_data_processing/usgs_sword_matchups_merritt_v2.parquet"
merritt_df.to_parquet(output_path, index=False)
print(f"Saved to: {output_path}")

## Gage Height

In [ ]:
with open('/work/pi_kandread_umass_edu/swot-urban/1_data_processing/all_wse_parts.pkl', 'rb') as f:
    all_wse_parts = pickle.load(f)

In [ ]:
# ============================================================================
# Setup Dask Distributed Client for Parallel Processing
# ============================================================================

# Initialize Dask client (adjust workers based on your system)
client = Client(n_workers=12, threads_per_worker=4, memory_limit='128GB')
print(f"✓ Dask Dashboard: {client.dashboard_link}")
print(f"  Workers: {len(client.scheduler_info()['workers'])}")

In [ ]:
# ============================================================================
# Process and combine gage height data
# ============================================================================

# Convert to Dask (not yet loaded into memory)
dask_dfs = []
for i, df_part in enumerate(all_wse_parts):
    dask_df = dd.from_pandas(df_part, npartitions=1)
    dask_dfs.append(dask_df)
    all_wse_parts[i] = None
    
    if i % 50 == 0:
        gc.collect()

# Concatenate with Dask (still not loaded into memory)
all_gage_dask = dd.concat(dask_dfs, axis=0)

In [ ]:
all_gage_dask.columns

In [ ]:
# Rename datetime column
if 'datetime' in all_gage_dask.columns:
    all_gage_dask = all_gage_dask.rename(columns={'datetime': 'measurement_dt'})
all_gage_dask['measurement_dt'] = dd.to_datetime(all_gage_dask['measurement_dt'])

# Find the gage height value column (usually named like '00065' or '00065_Mean')
gage_col = [col for col in all_gage_dask.columns if '00065' in col and 'cd' not in col.lower()][0]
all_gage_dask = all_gage_dask.rename(columns={gage_col: 'gage_height_ft'})

# Drop all quality code columns (columns ending with '_cd')
cd_columns = [col for col in all_gage_dask.columns if col.lower().endswith('_cd')]
if cd_columns:
    all_gage_dask = all_gage_dask.drop(columns=cd_columns)
    print(f"Dropped {len(cd_columns)} quality code columns (_cd)")
else:
    print("No quality code columns (_cd) found to drop")

In [ ]:
print("="*80)
print("Missing Gage Height Data Restructuring (Dask version)")
print("="*80)

# Step 1: Identify 00065_* columns
print("\nStep 1: Identifying 00065_* columns...")

all_columns = all_gage_dask.columns.tolist()
gage_height_columns = [col for col in all_columns if col.startswith('00065_')]

print(f"Total 00065_* columns found: {len(gage_height_columns)}")

# Step 2: Exclude datum-related columns
print("\nStep 2: Filtering out datum-related columns...")

datum_keywords = [
    # 'navd88', 'navd 1988', 'igld85', 'igld 85', 
    # 'nws datum', 'usgs/nws datum',
    # 'elevation referenced to'
]

columns_to_exclude = []
for col in gage_height_columns:
    col_lower = col.lower()
    if any(keyword in col_lower for keyword in datum_keywords):
        columns_to_exclude.append(col)

print(f"Columns to exclude (datum-related): {len(columns_to_exclude)}")
for col in sorted(columns_to_exclude):
    print(f"  - {col}")

working_columns = [col for col in gage_height_columns if col not in columns_to_exclude]
print(f"\nWorking with {len(working_columns)} columns after exclusion")

# Step 3: Identify paired columns
print("\nStep 3: Identifying paired columns (upstream/downstream, headwater/tailwater)...")

paired_columns = {
    'upstream_downstream': {'left': [], 'right': []},
    'headwater_tailwater': {'left': [], 'right': []},
}

unpaired_columns = []

for col in working_columns:
    col_lower = col.lower()
    if 'upstream' in col_lower:
        paired_columns['upstream_downstream']['left'].append(col)
    elif 'downstream' in col_lower or 'down stream' in col_lower:
        paired_columns['upstream_downstream']['right'].append(col)
    elif 'headwater' in col_lower:
        paired_columns['headwater_tailwater']['left'].append(col)
    elif 'tailwater' in col_lower:
        paired_columns['headwater_tailwater']['right'].append(col)
    else:
        unpaired_columns.append(col)

print(f"\nPaired columns found:")
for pair_type, cols in paired_columns.items():
    print(f"  {pair_type}:")
    print(f"    Left ({len(cols['left'])}): {cols['left']}")
    print(f"    Right ({len(cols['right'])}): {cols['right']}")

print(f"\nUnpaired columns ({len(unpaired_columns)}): will go to gage_height_missing")

# Step 4: Define processing function
print("\nStep 4: Creating processing function...")

def process_partition(partition_df, unpaired_cols, paired_cols):
    """Process each partition with pandas operations"""
    # Create new columns
    result_df = partition_df[['measurement_dt', 'site_no', 'gage_height_ft']].copy()
    result_df['gage_height_missing'] = np.nan
    result_df['gage_height_missing_left'] = np.nan
    result_df['gage_height_missing_right'] = np.nan
    result_df['source_column'] = ''
    
    # Process unpaired columns
    for col in unpaired_cols:
        if col not in partition_df.columns:
            continue
        mask = partition_df[col].notna()
        no_value_yet = result_df['gage_height_missing'].isna()
        update_mask = mask & no_value_yet
        if update_mask.sum() > 0:
            result_df.loc[update_mask, 'gage_height_missing'] = partition_df.loc[update_mask, col]
            result_df.loc[update_mask, 'source_column'] = col
    
    # Process paired columns - left (upstream/headwater)
    all_left_cols = paired_cols['upstream_downstream']['left'] + paired_cols['headwater_tailwater']['left']
    for left_col in all_left_cols:
        if left_col not in partition_df.columns:
            continue
        mask = partition_df[left_col].notna()
        no_value_yet = result_df['gage_height_missing_left'].isna()
        update_mask = mask & no_value_yet
        if update_mask.sum() > 0:
            result_df.loc[update_mask, 'gage_height_missing_left'] = partition_df.loc[update_mask, left_col]
            for idx in result_df[update_mask].index:
                existing = result_df.loc[idx, 'source_column']
                new_source = f"left:{left_col}"
                result_df.loc[idx, 'source_column'] = f"{existing},{new_source}" if existing else new_source
    
    # Process paired columns - right (downstream/tailwater)
    all_right_cols = paired_cols['upstream_downstream']['right'] + paired_cols['headwater_tailwater']['right']
    for right_col in all_right_cols:
        if right_col not in partition_df.columns:
            continue
        mask = partition_df[right_col].notna()
        no_value_yet = result_df['gage_height_missing_right'].isna()
        update_mask = mask & no_value_yet
        if update_mask.sum() > 0:
            result_df.loc[update_mask, 'gage_height_missing_right'] = partition_df.loc[update_mask, right_col]
            for idx in result_df[update_mask].index:
                existing = result_df.loc[idx, 'source_column']
                new_source = f"right:{right_col}"
                result_df.loc[idx, 'source_column'] = f"{existing},{new_source}" if existing else new_source
    
    return result_df

# Step 5: Apply processing to create lazy computation graph
print("\nStep 5: Creating computation graph...")

meta = pd.DataFrame({
    'measurement_dt': pd.Series(dtype='datetime64[ns]'),
    'site_no': pd.Series(dtype='object'),
    'gage_height_ft': pd.Series(dtype='float64'),
    'gage_height_missing': pd.Series(dtype='float64'),
    'gage_height_missing_left': pd.Series(dtype='float64'),
    'gage_height_missing_right': pd.Series(dtype='float64'),
    'source_column': pd.Series(dtype='object')
})

all_gage_miss_dask = all_gage_dask.map_partitions(
    process_partition,
    unpaired_columns,
    paired_columns,
    meta=meta
)

print(f"Computation graph created with {all_gage_miss_dask.npartitions} partitions")

In [ ]:
# Convert all feet-based gage height columns to meters and rename columns
# All gage_height columns are in feet - convert to meters
ft_cols = ['gage_height_ft', 'gage_height_missing', 
           'gage_height_missing_left', 'gage_height_missing_right']

for col in ft_cols:
    all_gage_miss_dask[col] = all_gage_miss_dask[col] * 0.3048
    print(f"✓ Converted {col} to meters")

# Rename gage_height_ft to gage_height_m
all_gage_miss_dask = all_gage_miss_dask.rename(columns={'gage_height_ft': 'gage_height_m'})
print("\n✓ Renamed gage_height_ft → gage_height_m")

print("\nFinal columns:", list(all_gage_miss_dask.columns))

### Datum Conversion

In [ ]:
all_gage_dask = all_gage_miss_dask.copy()
# n_partitions = 100  # Adjust based on your data size (50-200 usually good)
# all_gage_dask = all_gage_dask.repartition(npartitions=n_partitions)

In [ ]:
selected_12 = gpd.read_file('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/selected_12_vD.shp')
selected_12_site = selected_12.site_no.to_list()
all_gage_dask = all_gage_dask[all_gage_dask.site_no.isin(selected_12_site)]

In [ ]:
all_gage = all_gage_dask.compute()

In [ ]:
all_gage_df = all_gage.copy()

In [ ]:
# Priority order: gage_height_m > missing > missing_left > missing_right
def get_valid_gage_height(row):
    """Get first non-NaN value following priority order"""
    if pd.notna(row['gage_height_m']):
        return row['gage_height_m']
    elif pd.notna(row['gage_height_missing']):
        return row['gage_height_missing']
    elif pd.notna(row['gage_height_missing_left']):
        return row['gage_height_missing_left']
    elif pd.notna(row['gage_height_missing_right']):
        return row['gage_height_missing_right']
    else:
        return np.nan

all_gage_df['gage_height_valid'] = all_gage_df.apply(get_valid_gage_height, axis=1)

In [ ]:
print("NaN values? "+str(all_gage_df[all_gage_df.gage_height_valid.isna()].shape[0]))

In [ ]:
# ============================================================================
# Step 0: Prepare data and merge with site info
# ============================================================================
print("\n--- Step 0: Preparing data ---")

if not all_site_info.empty:
    # Prepare site info with needed columns
    site_info_subset = all_site_info[['alt_va', 'site_no', 'alt_datum_cd',\
                                      'dec_lat_va', 'dec_long_va','dec_coord_datum_cd',\
                                      'lat_va','long_va','coord_datum_cd',\
                                      ]].copy()
    
    # Convert site_no to string to match all_gage_df
    site_info_subset['site_no'] = site_info_subset['site_no'].astype(str)
    all_gage_df['site_no'] = all_gage_df['site_no'].astype(str)
    
    # Merge (Dask merge works the same way)
    all_gage_df = all_gage_df.merge(site_info_subset, on='site_no')#, how='left', broadcast = True)
    
    print(f"Merged site info with gage data")
else:
    print("Warning: No site info available for datum conversion")
    raise ValueError("Site info is required for datum conversion!")

In [ ]:
print("NaN values? "+str(all_gage_df[all_gage_df.alt_va.isna()].shape[0]))
# print("Which site? "+str(all_gage_df[all_gage_df.alt_va.isna()].site_no.iloc[0]))

#### Fill alt_va with DEM

In [ ]:
def get_dem_elevation(lat, lon, buffer=0.1):
    """Extract Elevation from DEM"""

    bbox = f"{lon-buffer},{lat-buffer},{lon+buffer},{lat+buffer}"

    datasets = [
        ('Digital Elevation Model (DEM) 1 meter', '1m'),
        ('National Elevation Dataset (NED) 1/9 arc-second', '~3m'),
        ('National Elevation Dataset (NED) 1/3 arc-second', '~10m'),
        ('National Elevation Dataset (NED) 1 arc-second', '~30m')
    ]

    for dataset_name, resolution in datasets:
        params = {
            'bbox': bbox,
            'datasets': dataset_name,
            'prodFormats': 'GeoTIFF',
            'max': 1
        }

        response = requests.get(
            'https://tnmaccess.nationalmap.gov/api/v1/products',
            params=params
        )

        data = response.json()

        if data['total'] > 0:
            download_url = data['items'][0]['downloadURL']
            dem_response = requests.get(download_url)

            with rasterio.open(BytesIO(dem_response.content)) as src:
                if not (src.bounds.left <= lon <= src.bounds.right and
                        src.bounds.bottom <= lat <= src.bounds.top):
                    continue

                crs_wkt = src.crs.to_wkt() if src.crs else ''
                vertical_datum = 'Unknown'

                if 'NAVD88' in crs_wkt or 'NAVD_1988' in crs_wkt:
                    vertical_datum = 'NAVD88'
                elif 'NGVD29' in crs_wkt or 'NGVD_1929' in crs_wkt:
                    vertical_datum = 'NGVD29'
                elif 'EPSG:5703' in str(src.crs):
                    vertical_datum = 'NAVD88'

                row, col = src.index(lon, lat)
                elevation = src.read(1)[row, col]

                if elevation != src.nodata:
                    info = {
                        'elevation': float(elevation),
                        'dataset': dataset_name,
                        'resolution': resolution,
                        'vertical_datum': vertical_datum,
                        'crs': str(src.crs)
                    }
                    return info

    return None

In [ ]:
missing_sites = all_gage_df[all_gage_df.alt_va.isna()].site_no.unique()

elevation_dict = {}
for site_no in tqdm(missing_sites):
    site_data = all_gage_df[all_gage_df.site_no == site_no].iloc[0]
    lat, lon = site_data.dec_lat_va, site_data.dec_long_va
    
    result = get_dem_elevation(lat, lon)
    if result:
        elevation_dict[site_no] = result['elevation']
    else:
        print(f"No DEM data for {site_no}")

for site_no, elevation in elevation_dict.items():
    mask = all_gage_df.site_no == site_no
    all_gage_df.loc[mask, 'alt_va'] = elevation
    all_gage_df.loc[mask, 'alt_datum_cd'] = 'NAVD88'

print(f"Filled {len(elevation_dict)} sites")
print(f"Remaining NaN: {all_gage_df.alt_va.isna().sum()}")

#### Continue...

In [ ]:
all_gage_df[['dec_coord_datum_cd', 'alt_datum_cd']].drop_duplicates()

In [ ]:
print("NaN values? "+str(all_gage_df[all_gage_df.alt_va.isna()].shape[0]))

In [ ]:
# Step 1: Calculate 0.1-year precision Epoch (Harlan et al. 2025 method)
def datetime_to_epoch_01yr(dt):
    """
    Convert datetime to 0.1-year precision epoch
    Following Harlan et al. (2025): "to the nearest 0.1 year"
    
    Examples:
    - 2024-01-15 -> 2024.0
    - 2024-03-15 -> 2024.2
    - 2024-06-20 -> 2024.5
    """
    year = dt.year
    day_of_year = dt.timetuple().tm_yday
    fraction = day_of_year / 365.25  # Account for leap years
    epoch = year + round(fraction * 10) / 10
    return epoch

# Apply to all observations (pandas DataFrame)
all_gage_df['epoch_01yr'] = all_gage_df['measurement_dt'].apply(datetime_to_epoch_01yr)
print(f"Calculated epochs for {len(all_gage_df):,} observations")

# Show epoch distribution
print(all_gage_df['epoch_01yr'].describe())
print(f"Unique epochs: {all_gage_df['epoch_01yr'].nunique()}")

In [ ]:
# Step 2: Create Site-Epoch Groups for Efficient VDatum Processing
# Group by site and epoch, calculate median WSE for each group
unique_site_epochs = all_gage_df.groupby(['site_no', 'epoch_01yr']).agg({
    'dec_lat_va': 'first',
    'dec_long_va': 'first',
    'dec_coord_datum_cd': 'first',
    'lat_va': 'first',
    'long_va': 'first',
    'coord_datum_cd': 'first',
    'alt_datum_cd': 'first',
    'alt_va': 'first',
}).reset_index()

print(f"Created {len(unique_site_epochs):,} unique (site × epoch) groups")

#### Datum Conversion with GDAL and Pyproj

- Step 1: lon/lat -> NAD83(2011)
- Step 2: alt_va_m (NGVD29) -> alt_navd88_m
- Step 3: datum_offset_m = EGM2008_alt - NAVD88_alt
- Step 4: wse_egm2008_m = gage_height_valid + alt_navd88_m + datum_offset_m 
- Step 5: wse_swot_compatible_m = wse_egm2008_m + mean_tide_correction_m 

In [ ]:
all_gage_df.columns

In [ ]:
all_gage_df[['alt_datum_cd','site_no']].drop_duplicates()

In [ ]:
# Step 1: NAD27/NAD83 -> NAD83(2011) (horizontal only, no epoch)
def horizontal_to_nad83_2011(df, use_decimal_coords=True):
    """NAD27/NAD83 -> NAD83(2011) - static transformation"""
    df = df.copy()
    
    if use_decimal_coords:
        lat_col, lon_col = 'dec_lat_va', 'dec_long_va'
        h_datum_col = 'dec_coord_datum_cd'
    else:
        lat_col, lon_col = 'lat_va', 'long_va'
        h_datum_col = 'coord_datum_cd'
    
    df['lon_2011'] = df[lon_col].astype(float)
    df['lat_2011'] = df[lat_col].astype(float)
    
    H_EPSG = {'NAD27': 4267, 'NAD83': 4269}
    
    for h_datum, src_epsg in H_EPSG.items():
        mask = df[h_datum_col] == h_datum
        if not mask.any():
            continue
        
        print(f"\nStep 1: {h_datum} → NAD83(2011): {mask.sum()} sites")
        
        coords = []
        indices = []
        for idx, row in df[mask].iterrows():
            coords.append(f"{row[lon_col]} {row[lat_col]} 0")
            indices.append(idx)
        
        cmd = [
            'gdaltransform',
            '-s_srs', f'EPSG:{src_epsg}',
            '-t_srs', 'EPSG:6318'
        ]
        
        result = subprocess.run(cmd, input="\n".join(coords), capture_output=True, text=True)
        
        for i, line in enumerate(result.stdout.strip().split('\n')):
            parts = line.split()
            if len(parts) >= 2:
                df.loc[indices[i], 'lon_2011'] = float(parts[0])
                df.loc[indices[i], 'lat_2011'] = float(parts[1])
    
    return df

# Step 2: NGVD29 -> NAVD88 (vertical only, static)
def ngvd29_to_navd88(df):
    """NGVD29 -> NAVD88 via VERTCON"""
    df = df.copy()
    df['alt_va_m'] = df['alt_va'] * 0.3048
    df['alt_navd88_m'] = df['alt_va_m'].copy()
    
    mask = df['alt_datum_cd'] == 'NGVD29'
    if not mask.any():
        print("\nStep 2: No NGVD29 sites")
        return df
    
    print(f"\nStep 2: NGVD29 → NAVD88: {mask.sum()} sites")
    
    coords = []
    indices = []
    for idx, row in df[mask].iterrows():
        coords.append(f"{row['lon_2011']} {row['lat_2011']} {row['alt_va_m']}")
        indices.append(idx)
    
    cmd = [
        'gdaltransform',
        '-s_srs', 'EPSG:6318+EPSG:7968',
        '-t_srs', 'EPSG:6318+EPSG:5703'
    ]
    
    result = subprocess.run(cmd, input="\n".join(coords), capture_output=True, text=True)
    
    for i, line in enumerate(result.stdout.strip().split('\n')):
        parts = line.split()
        if len(parts) >= 3:
            df.loc[indices[i], 'alt_navd88_m'] = float(parts[2])
    
    return df
    
# Step 3: EPSG:6349 -> SWOT with epoch
def to_swot_crs(df):
    """
    NAD83(2011)+NAVD88 -> WGS84(G1762)+EGM2008
    - source_epoch: 2010.0 (fixed)
    - target_epoch: epoch_01yr (per observation)
    """
    df = df.copy()
    df['datum_offset_m'] = None
    
    print(f"\nStep 3: EPSG:6349 → EPSG:9057+EPSG:3855")
    
    # Group by target epoch
    for epoch_yr, group in df.groupby('epoch_01yr'):
        coords = []
        indices = []
        for idx, row in group.iterrows():
            coords.append(f"{row['lon_2011']} {row['lat_2011']} {row['alt_navd88_m']}")
            indices.append(idx)
        
        cmd = [
            'gdaltransform',
            '-s_coord_epoch', '2010.0',
            '-t_coord_epoch', str(float(epoch_yr)),
            '-s_srs', 'EPSG:6349',
            '-t_srs', 'EPSG:9057+EPSG:3855'
        ]
        
        result = subprocess.run(cmd, input="\n".join(coords), capture_output=True, text=True)
        
        for i, line in enumerate(result.stdout.strip().split('\n')):
            parts = line.split()
            if len(parts) >= 3:
                idx = indices[i]
                df.loc[idx, 'lon_wgs84'] = float(parts[0])  # 추가
                df.loc[idx, 'lat_wgs84'] = float(parts[1])  # 추가
                alt_out = float(parts[2])
                df.loc[idx, 'datum_offset_m'] = alt_out - df.loc[idx, 'alt_navd88_m']
        
        print(f"  epoch={epoch_yr}: {len(indices)} sites")
    
    offsets = df['datum_offset_m'].dropna()
    print(f"n={len(offsets)}, offset: {offsets.mean():.3f}m [{offsets.min():.3f}, {offsets.max():.3f}]")
    
    return df

In [ ]:
df = horizontal_to_nad83_2011(unique_site_epochs)

In [ ]:
df = ngvd29_to_navd88(df)
df = to_swot_crs(df)

#### Vdatum working well, but only for coastal area

In [ ]:
# ============================================================================
# Step 3: VDatum Processing
# ============================================================================

VDATUM_DIR = "/project/pi_kandread_umass_edu/swot-urban/vdatum"
JAVA_MODULE = "openjdk/11.0.24"
module = "openjdk/11.0.24"

def vdatum_convert_point(
    lon,
    lat,
    z_navd88,
    epoch_in=2010.0,
    epoch_out=2023.9,
    vertical_unit="m",
    input_vert="NAVD88",
    input_hori="NAD83_2011",
    deg2dms=False,
    region=4,
    return_stdout=True,
):

    deg2dms_flag = "-deg2dms " if deg2dms else ""

    cmd = (
        f"module load {JAVA_MODULE} && "
        "java -jar vdatum.jar "
        # "ihorz:NAD83_2011:geo:deg "
        # f"ivert:NAVD88:{vertical_unit}:height:geoid18 "
        f"ihorz:{input_hori}:geo:deg "
        f"ivert:{input_vert}:{vertical_unit}:height:geoid18 "
        "ohorz:WGS84_G1674:geo:deg "
        "overt:EGM2008:m:height:EGM2008 "
        f"{deg2dms_flag}"
        f"-pt:{lon:.8f},{lat:.8f},{z_navd88:.3f} "
        f"epoch:{epoch_in:.1f}:{epoch_out:.1f} "
        f"region:{region}"
    )

    result = subprocess.run(
        cmd,
        shell=True,
        executable="/bin/bash",
        cwd=VDATUM_DIR,     
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        raise RuntimeError(
            "VDatum command failed\n"
            f"Command:\n{cmd}\n\n"
            f"STDOUT:\n{result.stdout}\n\n"
            f"STDERR:\n{result.stderr}"
        )

    if return_stdout:
        return result.stdout
    else:
        print(result.stdout)
        return None

In [ ]:
lon = unique_site_epochs.dec_long_va[0]
lat = unique_site_epochs.dec_lat_va[0]
height_navd88 = unique_site_epochs.median_wse_navd88_m[0]
epoch_out = unique_site_epochs.epoch_01yr[0]

coord_datum_map = {"NAD83": "NAD83_2011", "NAD27": "NAD27"}
coord_datum = unique_site_epochs.coord_datum_cd[0]
input_hori = coord_datum_map.get(coord_datum, np.nan)

alt_datum_map = {"NAVD88": "NAVD88", "NGVD29": "NGVD29"}
alt_datum = unique_site_epochs.alt_datum_cd[0]
input_vert = alt_datum_map.get(alt_datum, np.nan)

epoch_in = 2010.0             

stdout = vdatum_convert_point(
    lon=lon,
    lat=lat,
    z_navd88=height_navd88,
    input_vert=input_vert,
    input_hori=input_hori,
    epoch_in=epoch_in,
    epoch_out=epoch_out,
    vertical_unit="m",
    deg2dms=False,
    region=4,
)

print(stdout)

In [ ]:
VDATUM_DIR = "/project/pi_kandread_umass_edu/swot-urban/vdatum"
JAVA_MODULE = "openjdk/11.0.24"
module = "openjdk/11.0.24"

def vdatum_convert_point(
    lon,
    lat,
    z_navd88,
    epoch_in=2010.0,
    epoch_out=2023.9,
    vertical_unit="m",
    input_vert="NAVD88",
    input_hori="NAD83_2011",
    deg2dms=False,
    region=4,
    return_stdout=True,
):

    deg2dms_flag = "-deg2dms " if deg2dms else ""

    cmd = (
        f"module load {JAVA_MODULE} && "
        "java -jar vdatum.jar "
        # "ihorz:NAD83_2011:geo:deg "
        # f"ivert:NAVD88:{vertical_unit}:height:geoid18 "
        f"ihorz:{input_hori}:geo:deg "
        f"ivert:{input_vert}:{vertical_unit}:height:geoid18 "
        "ohorz:WGS84_G1674:geo:deg "
        "overt:EGM2008:m:height:EGM2008 "
        f"{deg2dms_flag}"
        f"-pt:{lon:.8f},{lat:.8f},{z_navd88:.3f} "
        f"epoch:{epoch_in:.1f}:{epoch_out:.1f} "
        f"region:{region}"
    )

    result = subprocess.run(
        cmd,
        shell=True,
        executable="/bin/bash",
        cwd=VDATUM_DIR,     
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        raise RuntimeError(
            "VDatum command failed\n"
            f"Command:\n{cmd}\n\n"
            f"STDOUT:\n{result.stdout}\n\n"
            f"STDERR:\n{result.stderr}"
        )

    if return_stdout:
        return result.stdout
    else:
        print(result.stdout)
        return None

def parse_vdatum_output(stdout):
    for line in stdout.split('\n'):
        if 'Height/Z:' in line:
            numbers = re.findall(r'-?\d+\.\d+', line)
            if len(numbers) >= 2:
                return float(numbers[1])
    raise ValueError("No Height/Z found")

# Check for existing progress
checkpoint_file = '/tmp/vdatum_progress_ver3.parquet'
if os.path.exists(checkpoint_file):
    completed = pd.read_parquet(checkpoint_file)
    print(f"Resuming from checkpoint: {len(completed)} completed")
    remaining = unique_site_epochs[~unique_site_epochs.index.isin(completed.index)]
else:
    completed = pd.DataFrame()
    remaining = unique_site_epochs

results = []
total = len(remaining)
print(f"Processing {total} points...")

coord_datum_map = {"NAD83": "NAD83_2011", "NAD27": "NAD27"}
alt_datum_map = {"NAVD88": "NAVD88", "NGVD29": "NGVD29"}

for i, (idx, row) in enumerate(remaining.iterrows(), 1):
    try:
        lon = row['dec_long_va']
        lat = row['dec_lat_va']
        height_navd88 = row['median_wse_navd88_m']
        epoch_out = row['epoch_01yr']
        input_hori = coord_datum_map.get(row['coord_datum_cd'], np.nan)
        input_vert = alt_datum_map.get(row['alt_datum_cd'], np.nan)
        epoch_in = 2010.0
        
        egm2008_height = None
        for test_region in [4] + list(range(1, 14)):
            try:
                stdout = vdatum_convert_point(
                    lon=lon, lat=lat, z_navd88=height_navd88,
                    input_vert=input_vert, input_hori=input_hori,
                    epoch_in=epoch_in, epoch_out=epoch_out,
                    vertical_unit="m", deg2dms=False, region=test_region
                )
                egm2008_height = parse_vdatum_output(stdout)
                break
            except:
                continue
        
        if egm2008_height is None:
            raise ValueError("No valid region")
        
        results.append({
            'site_no': row['site_no'], 
            'epoch_01yr': epoch_out,
            'geoid_offset_m': egm2008_height - height_navd88
        })
        
    except Exception as e:
        print(f"Failed {row['site_no']}: {str(e)[:50]}")
        results.append({
            'site_no': row['site_no'], 
            'epoch_01yr': row['epoch_01yr'],
            'geoid_offset_m': np.nan
        })
    
    if i % 10 == 0:
        checkpoint = pd.DataFrame(results)
        checkpoint.to_parquet(checkpoint_file)
        print(f"{i}/{total} ({i/total*100:.0f}%)")

# Final merge
transforms = pd.concat([completed, pd.DataFrame(results)])
unique_site_epochs = unique_site_epochs.merge(
    transforms[['site_no', 'epoch_01yr', 'geoid_offset_m']], 
    on=['site_no', 'epoch_01yr'], how='left'
)
os.remove(checkpoint_file)

In [ ]:
all_gage_df = all_gage_df.merge(
    unique_site_epochs[['site_no', 'epoch_01yr', 'geoid_offset_m']],
    on=['site_no', 'epoch_01yr'], how='left'
)
all_gage_df['wse_egm2008_m'] = all_gage_df['wse_navd88_m'] + all_gage_df['geoid_offset_m']

all_gage_df.to_parquet(
    '/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/usgs_gage_vdatum.parquet',
    compression='snappy', index=False
)
print(f"Done: {all_gage_df['wse_egm2008_m'].notna().sum():,}")

#### VDatum REST API

In [ ]:
# lon = unique_site_epochs.dec_long_va[0]
# lat = unique_site_epochs.dec_lat_va[0]
# height_navd88 = unique_site_epochs.median_wse_navd88_m[0]
# epoch_out = unique_site_epochs.epoch_01yr[0]
# print(lon, lat, height_navd88, epoch_out)

# params = {
#     "region": "CONTIGUOUS", "s_h_frame": "NAD83_2011", "s_coor": "geo",
#     "s_h_unit": "", "s_h_zone": "", "s_v_frame": "NAVD88", "s_v_unit": "m",
#     "s_v_elevation": "height", "s_v_geoid": "geoid18",
#     "s_x": str(lon), "s_y": str(lat), "s_z": str(height_navd88),
#     "t_h_frame": "WGS84_G1674", "t_coor": "geo", "t_h_unit": "", "t_h_zone": "",
#     "t_v_frame": "EGM2008", "t_v_unit": "m", "t_v_elevation": "height",
#     "t_v_geoid": "egm2008", "epoch_in": 2010, "epoch_out": str(epoch_out)
# }

# url = "https://vdatum.noaa.gov/vdatumweb/api/convert"

# print("Testing VDatum API with your Java command parameters...")
# print(f"Epochs: {params['epoch_in']} -> {params['epoch_out']}")

# try:
#     response = requests.get(url, params=params, timeout=30)
#     print(f"\nStatus: {response.status_code}")
    
#     if response.status_code == 200:
#         data = response.json()
        
#         if 'errorCode' in data:
#             print(f" API Error {data['errorCode']}: {data.get('message', '')}")
#         else:
#             print(f"\nSuccess!")
            
#             input_m = float(data['s_z'])
#             output_m = float(data['t_z'])
#             print(f"  Geoid offset: {output_m - input_m:.3f} m")
            
#     else:
#         print(f"HTTP Error: {response.status_code}")
#         print(response.text)
        
# except Exception as e:
#     print(f"Error: {e}")

In [ ]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import time
import pandas as pd
import numpy as np

class VDatumAPITransformer:
    def __init__(self, request_delay=2.0):
        self.base_url = "https://vdatum.noaa.gov/vdatumweb/api/convert"
        self.request_delay = request_delay
        session = requests.Session()
        retry = Retry(total=3, status_forcelist=[500, 502, 503, 504], 
                     allowed_methods=["GET"], backoff_factor=3)
        session.mount("https://", HTTPAdapter(max_retries=retry))
        self.session = session
    
    def transform_point(self, lon, lat, height_navd88, epoch_out, epoch_in=2010.0):
        params = {
            "region": "CONTIGUOUS", "s_h_frame": "NAD83_2011", "s_coor": "geo",
            "s_h_unit": "", "s_h_zone": "", "s_v_frame": "NAVD88", "s_v_unit": "m",
            "s_v_elevation": "height", "s_v_geoid": "geoid18",
            "s_x": str(lon), "s_y": str(lat), "s_z": str(height_navd88),
            "t_h_frame": "WGS84_G1674", "t_coor": "geo", "t_h_unit": "", "t_h_zone": "",
            "t_v_frame": "EGM2008", "t_v_unit": "m", "t_v_elevation": "height",
            "t_v_geoid": "egm2008", "epoch_in": str(epoch_in), "epoch_out": str(epoch_out)
        }
        
        try:
            time.sleep(self.request_delay)
            response = self.session.get(self.base_url, params=params, timeout=30)
            
            if response.status_code == 429:
                time.sleep(60)
                response = self.session.get(self.base_url, params=params, timeout=30)
            
            response.raise_for_status()
            data = response.json()
            
            if 'errorCode' in data:
                return {'success': False, 'egm2008_tide_free_m': np.nan, 
                       'geoid_offset_m': np.nan, 'uncertainty_m': np.nan,
                       'error': f"API Error {data.get('errorCode')}: {data.get('message')}"}
            
            egm2008_height = float(data.get('t_z', -999999))
            if egm2008_height == -999999.0:
                return {'success': False, 'egm2008_tide_free_m': np.nan,
                       'geoid_offset_m': np.nan, 'uncertainty_m': np.nan,
                       'error': 'Outside coverage'}
            
            return {
                'success': True, 'egm2008_tide_free_m': egm2008_height,
                'geoid_offset_m': egm2008_height - height_navd88,
                'uncertainty_m': float(data.get('uncertainty', np.nan)), 'error': None
            }
        except Exception as e:
            return {'success': False, 'egm2008_tide_free_m': np.nan,
                   'geoid_offset_m': np.nan, 'uncertainty_m': np.nan, 'error': str(e)}
    
    def transform_sequential(self, site_epochs_df, epoch_in=2010.0):
        results = []
        failed_points = []
        total = len(site_epochs_df)
        print(f"Processing {total} points (est. {total*self.request_delay/60:.1f} min)")
        
        for i, (idx, row) in enumerate(site_epochs_df.iterrows(), 1):
            result = self.transform_point(
                row['dec_long_va'], row['dec_lat_va'],
                row['median_wse_navd88_m'], row['epoch_01yr'], epoch_in
            )
            result.update({'site_no': row['site_no'], 'epoch_01yr': row['epoch_01yr']})
            results.append(result)
            
            if not result['success']:
                failed_points.append({
                    'index': i, 'site_no': row['site_no'], 'epoch': row['epoch_01yr'],
                    'lon': row['dec_long_va'], 'lat': row['dec_lat_va'], 
                    'error': result['error']
                })
                print(f"⚠️  Failed {i}/{total}: Site {row['site_no']}, {result['error'][:50]}")
            
            if i % 50 == 0 or i == total:
                print(f"{i}/{total} ({i/total*100:.0f}%)")
        
        if failed_points:
            print(f"\n⚠️  {len(failed_points)} points failed:")
            for fp in failed_points[:5]:
                print(f"  Site {fp['site_no']}: {fp['error'][:60]}")
            if len(failed_points) > 5:
                print(f"  ... and {len(failed_points)-5} more")
        
        return pd.DataFrame(results), pd.DataFrame(failed_points)

transformer = VDatumAPITransformer(request_delay=2.0)
site_epoch_transforms, failed_points_df = transformer.transform_sequential(unique_site_epochs, epoch_in=2010.0)

successful = site_epoch_transforms['success'].sum()
print(f"\n✓ {successful}/{len(site_epoch_transforms)} successful ({successful/len(site_epoch_transforms)*100:.1f}%)")

if successful > 0:
    valid = site_epoch_transforms[site_epoch_transforms['success']]
    print(f"Geoid offset: {valid['geoid_offset_m'].mean():.3f}±{valid['geoid_offset_m'].std():.3f} m")

all_gage_df = all_gage_df.merge(
    site_epoch_transforms[['site_no', 'epoch_01yr', 'geoid_offset_m', 'uncertainty_m']],
    on=['site_no', 'epoch_01yr'], how='left'
)
all_gage_df['wse_egm2008_m'] = all_gage_df['wse_navd88_m'] + all_gage_df['geoid_offset_m']

output_path = '/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/usgs_gage_vdatum_intermediate.parquet'
all_gage_df.to_parquet(output_path, compression='snappy', index=False)
print(f"✓ Saved: {output_path}")

if len(failed_points_df) > 0:
    failed_path = '/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/vdatum_failed_points.csv'
    failed_points_df.to_csv(failed_path, index=False)
    print(f"✓ Failed points saved: {failed_path}")

#### Continue...

In [ ]:
df_needed = df[['site_no', 'epoch_01yr', 'alt_navd88_m', 'datum_offset_m', 'lon_wgs84', 'lat_wgs84']]
all_gage_df = all_gage_df.merge(df_needed, on = ['site_no','epoch_01yr'])

In [ ]:
# Step 4: Convert gage height to absolute elevation (m)
all_gage_df['wse_egm2008_m'] = all_gage_df['gage_height_valid'] + all_gage_df['alt_navd88_m'] + all_gage_df['datum_offset_m']

In [ ]:
# Step 5: Tide-free -> Mean-tide correction (SWOT User Handbook)
print("Δh = (1 + k₂)H_perm √(5/4π) (3/2 sin²θ - 1/2)")

# Calculate latitude in radians
all_gage_df['latitude_rad'] = np.radians(all_gage_df['lat_wgs84'])

# Mean-tide correction
H_perm = -0.31460  # m
k2 = 0.3
all_gage_df['mean_tide_correction_m'] =\
(1 + k2) * H_perm * np.sqrt(5/(4*np.pi)) * (1.5*np.sin(all_gage_df['latitude_rad'])**2 - 0.5)

print(f"  Correction statistics:")
print(f"    Mean: {all_gage_df['mean_tide_correction_m'].mean():.4f} m")
print(f"    Range: [{all_gage_df['mean_tide_correction_m'].min():.4f}, "
      f"{all_gage_df['mean_tide_correction_m'].max():.4f}] m")

# Final SWOT-compatible WSE (EGM2008 mean-tide)
all_gage_df['wse_swot_compatible_m'] = (
    all_gage_df['wse_egm2008_m'] - all_gage_df['mean_tide_correction_m']
)

print(f"✓ Applied mean-tide correction to {all_gage_df['wse_swot_compatible_m'].notna().sum():,} observations")

# Verification
print("\n--- Verification ---")
print(f"Average correction: {all_gage_df['mean_tide_correction_m'].mean():.4f} m")
print(f"WSE difference (mean-tide - tide-free): "
      f"{(all_gage_df['wse_swot_compatible_m'] - all_gage_df['wse_egm2008_m']).mean():.4f} m")

### Relative WSE

In [ ]:
# Step 6: Relative WSE (median-normalized)
print("\n--- Step 6: Create Relative WSE ---")

all_gage_df['wse_usgs_relative_m'] = all_gage_df.groupby('site_no')['gage_height_valid'].transform(
    lambda x: x - x.median()
)

print(f"Created wse_relative_m for {all_gage_df['wse_usgs_relative_m'].notna().sum():,} observations")
print(f"  Mean: {all_gage_df['wse_usgs_relative_m'].mean():.4f} m (should be ~0)")

In [ ]:
# Save to parquet
output_path = '/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/usgs_gage_swot_compatible_gdal.parquet'
os.makedirs(os.path.dirname(output_path), exist_ok=True)

all_gage_df.to_parquet(output_path, compression='snappy', index=False)

print(f"✓ Saved: {output_path}")
print(f"  Shape: {all_gage_df.shape}")